# Importing Libraries

In [ ]:
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import gspread
import gspread_dataframe as gd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

no_regex_write= gc.open_by_url("GOOGLE_SHEET_URL").worksheet('no_regex')
correction_write= gc.open_by_url("GOOGLE_SHEET_URL").worksheet('correction')
final_models_write= gc.open_by_url("GOOGLE_SHEET_URL").worksheet('final models')

#template to write in a sheet
#gd.set_with_dataframe(sheetName_write, dataframe,row=1,col=1,include_index=False,include_column_header=True)
#temaplate to read a sheet
#df = gc.open('main sheet name')
#sheet = df.worksheet('no_regex')
#data = pd.DataFrame(sheet.get_all_records())



In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


analytics_mongo = client_db.get_database('analytics_db')

db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')

wave=app_mongo.get_database('app_db')



# DATA AQUISITION

In [ ]:
vahan_api_scrape=pd.read_csv("vahaan_scrape_Api.csv")
gps_vehicles=pd.read_sql("select vehicle_number from vehicles where id in (select vehicle_id from operator_vehicles where deboarding_time is null and is_active=True)",mysql_vds)      
api_scrape=vahan_api_scrape[vahan_api_scrape['Registration No'].isin(gps_vehicles['vehicle_number'])]  



In [ ]:
my_data=pd.read_csv("mydatavahan.csv")
my_data=my_data[~my_data['Registration No:'].isin(api_scrape['Registration No'])]
len(my_data)

In [ ]:
sheet_data=pd.read_excel("veh_data.xlsx")
sheet_data=sheet_data[~sheet_data['Number'].isin(api_scrape['Registration No'])]
len(sheet_data)

In [ ]:
mongo_data=pd.read_csv("mongo_data_vahan.csv")
mongo_data=mongo_data[~mongo_data['registration_no'].isin(api_scrape['Registration No'])]
len(mongo_data)

In [ ]:
my_data_list=my_data[['Registration No:','Maker / Model:']].values.tolist()

In [ ]:
sheet_list=sheet_data[['Number','Model']].values.tolist()

In [ ]:
mongo_list=mongo_data[['registration_no','maker_model']].values.tolist()

In [ ]:
vahan_api_list=api_scrape[['Registration No','Maker/Model']].values.tolist()

In [ ]:
total_list=my_data_list+sheet_list+mongo_list+vahan_api_list

In [ ]:
make_model=pd.DataFrame(total_list,columns=['vehicle_number','make/model'])

In [ ]:
make_model['make/model'].tolist()

# MAKE MODEL DATAFRAME

In [ ]:
make_model=make_model[make_model['make/model'].notnull()]

In [ ]:
make_model.drop_duplicates('vehicle_number',inplace=True)

In [ ]:
make_model['type']=make_model['make/model'].apply(lambda x: str(type(x)))
make_model['type'].value_counts()
make_model=make_model[make_model['type']!="<class 'datetime.datetime'>"]
del make_model['type']

In [ ]:
make_model['make']=make_model['make/model'].apply(lambda x : x.split('/')[0])
make_model['contains_upto']=make_model['make/model'].str.findall('upto').apply(len)
make_model['contains_upto'].value_counts()
make_model=make_model[make_model['contains_upto']==0]
del make_model['contains_upto']

## GETTING MAKE

In [ ]:
make_model['make']=make_model['make/model'].apply(lambda x : x.split('/')[0])
make_model['contains_numbers']=make_model['make'].str.findall(r'\d\d').apply(len)
make_model['contains_numbers'].value_counts()
make_model=make_model[make_model['contains_numbers']==0]
del make_model['contains_numbers']

## removing empty makes

In [ ]:
make_model=make_model[~make_model['make'].isin([' ','  ','   ','    ','     '])]

## finding makes using fuzzy

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

def OEM_categorisation(x):
    name_list=[ 'ASHOK LEYLAND',
        'TATA',
 'VE COMMERCIAL',
 'MAHINDRA',
 'EICHER',
 'DAIMLER',
 'SML ISUZU',
 'VECTRA',
 'Baldev',
 'FORCE',
 'PIAGGIO',
 'MAN TRUCKS',
 'AMW',
 'GENERAL']
    for i in name_list:
        if fuzz.partial_ratio(x,i)>70:
            #print('compared ' + i + ' with '+ x +' and SELECTED')w
            return i 
        else:
            None
            #print('compared ' + i + ' with '+ x +' and rejected')
    return x
                
make_model['sanitized_make']=make_model.make.apply(OEM_categorisation)    
make_model.head(2)

## GETTING MODEL

In [ ]:
def split_data(x):
    try:
        a=x.split('/')[1]
        return a
    except:
        a=None
        return a
make_model['model']=make_model['make/model'].apply(split_data)
make_model.head(2)

## removing vehicles with no model

In [ ]:
make_model['model'].notnull().value_counts()
make_model=make_model[make_model['model'].notnull()]
make_model.head(2)

# APPLYING REGEX ON MODELS

In [ ]:
make_model['model_regex']=make_model['model'].str.findall(r'(\d\d\d\d|407|ACE|\d\d.\d\d|DOST|BOLERO|BLAZO\d\d|BLAZO \d\d|709|909|207|912)')
make_model['model_regex']=make_model['model_regex'].apply(lambda x: x if len(x)>0 else None)
make_model['model_regex'].isna().value_counts()
make_model.head(2)

## choosing only select makes

In [ ]:
select_makes=make_model[make_model['sanitized_make'].isin(['MAHINDRA','TATA','ASHOK LEYLAND','VE COMMERCIAL','EICHER'])][make_model['model_regex'].notnull()]       
make_model.vehicle_number.nunique()
select_makes.vehicle_number.nunique()


## checking which patterns were missed

In [ ]:
missed_regex=pd.DataFrame(select_makes[select_makes['model_regex'].isna()].model.value_counts())
gd.set_with_dataframe(no_regex_write, missed_regex,row=1,col=1,include_index=False,include_column_header=True)
select_makes[select_makes['model_regex'].isna()].model.value_counts().keys().tolist()[0:10]


In [ ]:
select_makes['len_of_regex']=select_makes['model_regex'].apply(len)

In [ ]:
def list_string_merge(x):
    y=''
    for i in x:
        
        y+=" " + i
    return y
select_makes['model_list_merged']=select_makes['model_regex'].apply(list_string_merge)


## removing dots and spaces 

In [ ]:
import re
select_makes['removing_dot']=select_makes.model_list_merged.apply(lambda x: re.sub('\.', '', x) )
select_makes['removing_dot']=select_makes.removing_dot.apply(lambda x: re.sub('-', '', x) )
select_makes['removing_dot']=select_makes.removing_dot.apply(lambda x: re.sub(' ', '', x) )
select_makes['removing_dot']=select_makes.removing_dot.apply(lambda x: x[1:])


## get all the sheets data

In [ ]:
df = gc.open('MAKE MODELS')
sheet = df.worksheet('correction')
corrected_data = pd.DataFrame(sheet.get_all_records())
corrected_data['models']=corrected_data['models'].apply(str)
corrected_data['correction']=corrected_data['correction'].apply(str)

sheet = df.worksheet('final models')
final_models= pd.DataFrame(sheet.get_all_records())
final_models['final models']=final_models['final models'].apply(str)

## add any corrected values to final values

In [ ]:
new_corrected_models=corrected_data[~corrected_data['correction'].isin(final_models['final models'])][corrected_data['correction']!='']['correction'].unique()   
complete_new_final_models=pd.DataFrame(final_models['final models'].tolist()+new_corrected_models.tolist(),columns=['final models'])
gd.set_with_dataframe(final_models_write, complete_new_final_models,row=1,col=1,include_index=False,include_column_header=True)


## replace acording to correction

In [ ]:
correction_values=dict(corrected_data.values.tolist())

In [ ]:
select_makes['correction_model']=select_makes['removing_dot'].replace(correction_values)

## Compare current data with final models 

In [ ]:
def fuzzy_comparision(model_string):
    for i in final_models['final models'].value_counts().keys():
        if fuzz.partial_ratio(model_string,i)>95:
            return i
    return model_string

In [ ]:
select_makes['fuzzy_corrected_models']=select_makes['correction_model'].apply(fuzzy_comparision)

## after fuzzy check with final models,adding models that are not there in corrected data, to correction table in sheets

In [ ]:
new_correction_models=select_makes[~select_makes['fuzzy_corrected_models'].isin(corrected_data['models'])]['fuzzy_corrected_models'].unique().tolist()
new_correction_models=[[i, None] for i in new_correction_models]
new_correction_models=pd.DataFrame(new_correction_models,columns=['models','correction'])
new_correction_models=pd.concat([corrected_data,new_correction_models])
gd.set_with_dataframe(correction_write, new_correction_models,row=1,col=1,include_index=False,include_column_header=True)


In [ ]:
select_makes['fuzzy_corrected_models'].value_counts()

In [ ]:
a="""4018/4019
3118
4018
1212/1214
4028
4928
1611/1612/1613/1616/1618
2214/2515/2516/2518/2523
1611/1612/1613/1616/1618
2214/2515/2516/2518/2523
2516/2518
2518/2515
2523 / 2528
3516/3518
3516/3518
4018/4019/4028
4018/4019/4028/4021
4923 / 4928
AL2513 / AL2514 / AL2516 / AL2523 / AL2614 / AL2518
LPK2518
LPS 3516/3518
LPS 3516/3518
LPT1613/1616
LPT2518/2515/2516
LPT3118
BLAZO37
1109/1112
3118/3116/3123
909/912
Cummins Engine- 2515/2516/2518/2523
LPT-1612/1613/1616/1615
SE-1612/1613/1616/1615
SFC/LPT- 407/412 CNG
SFC/LPT- 709/712
LPK -1612/1613/1616
2516
3718
3718
3718
1611/1612/1613/1616
1611/1612/1613/1616
1611/1612/1613/1616
2515/2516/2518/2523
2515/2516/2518/2523
2515/2516/2518/2523
3116/3118/3123
3116/3118/3123
3116/3118/3123
3116/3118/3123
3116/3118/3123/3523
3516 / 3518
3718/U3718
4018 / 4019
4018/4019/4023/4028
4921/4923/4928
4928 / 4923
CO-3/10/1611/1612/1613/1616
DOST
GURU
914
1214
1617
3123
3723
3723
3723
3723
1617/1623
2523/2528
3123 / 3128
3123 / 3128
3123 / 3128
4023 / 4028 
20.16
20.16
30.25
3015
4035
4040
10.50/10.55/10.59
10.50/10.55/10.59 CNG
10.70/10.75/10.80
10.70/10.75/10.80 CNG
10.90/10.95
11.10/11.12/11.14
35.31/ 3025/3325
35.31/ 3025/3325
3531/30.18
3531/3025
4040/35/49/23
PRO 1095 / PRO 1090 / PRO 1080
PRO 1110/1114
PRO 3013/3016
Pro 6025
Pro 6031
Pro 6031
Pro 6031
Pro 6031
Pro 6037
Pro 6037
Pro 6037
Pro 6037
PRO10.49/50/55/59
PRO10.75/10.80
T-16
T-25
GENIO
Maximmo
Blazo 25
Blazo 25
Blazo 25
Blazo 25
Blazo 37
Blazo 37
Blazo 37
Blazo 40
Blazo 40
Blazo 49
Blazo 49
DI 3200
Load King
M&M BMT All Variants
M&M Camper All Variants
M&M Pick Up All Variants
T/Blazo 40
TORRO-25
TRACO 35
Truxo 25
Truxo31
31280
2522/28/18
25220/280
25280 RMC
31220/31280 
31280 RMC
4020/4028 /22
4928 /22
Samrat /SUPER
SAMRAT CNG
SARTAJ CNG
SARTAJ/PRESTIGE/SUPREME
SUPER CNG
207
Lpk 407
Lpk 909
2518
3518
3516 / 3518
4018/4019/4023/4028
407 PICK UP
4921/4923/4928
4923 signa Series
4928 / 4923
909/912 CNG/Non-CNG
ACE HT/HD/EX
ACE HT/HD/EX CNG
ACE SUPER
ACE ZIP
ACE ZIP CNG
LP1510/1512
Lpk 3118
LPT1412 CRX BS4
LPT-1612/1613/1616
LPT-1612/1613/1616/1510/1512
LPT-1612/1613/1616/1510/1512/1615
LPT3718/LPT3723
LPT3718/LPT3723
LPT3718/LPT3723
PRIMA 4023/28
PRIMA4921 / PRIMA4923 / PRIMA 4928
LPk 3818
SE-1616/1613
SFC/LPT- 407/412
Turbo Engine-2515/2516/2518/2523
Cummings Engine-2515/2516/2518/2523
Turbo Engine-2515/2516/2518/2523
Cummings Engine-2515/2516/2518/2523
Cummings Engine-2515/2516/2518/2523
3718
FM 12- 380-6x4 3 Axle
FM 12 460 PULLER 3 Axle
FM 400-PULLER 3 Axle
AL 3118
Al 3718
SARTAJ TIPPER
SUPER TIPPER
SUPREME  - TIPPER """ 

In [ ]:
a=a.splitlines()

In [ ]:
final=[]
for i in a:
    j=i.split('/')
    final.append(j)

In [ ]:
final_1=[]
for i in final:
    for j in i:
        final_1.append(j)

In [ ]:
final=list(dict.fromkeys(final_1))


In [ ]:
len(final_1)

In [ ]:
final